# LLM Classification Finetuning: Inference + Benchmark notebook (llm-cls-infer)

**Settings:** Accelerator = **GPU T4 x2**, Internet = **OFF** (submission-ku kandippa OFF)

**Inputs (Add Input):**
1. Competition data (LLM Classification Finetuning)
2. Your Work → Notebooks → **llm-cls-train** (training output: `qwen-cls/`, `val.parquet`, `train_info.json`, `wheels/`)

**Enna pannuthu:**
- Same fine-tuned model-a pala inference settings-la run panni measure pannuthu: speed (sec, samples/sec), memory (VRAM), quality (val log loss).
- Variants: fp32 naive → fp16 → batching → length-sorted batching → INT8/INT4 (bitsandbytes) → 2 GPU → swap TTA.
- Kadaisila best setting-la test predict panni `submission.csv`.

Submission rerun-la (hidden test) benchmarks skip aagum, prediction mattum odum.

In [ ]:
import os, glob, json, time, subprocess, threading
import numpy as np, pandas as pd, torch
from sklearn.metrics import log_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"input-la kedaikala: {pattern} (Add Input pannina?)"
    return hits[0]

MODEL_DIR = os.path.dirname(find("qwen-cls/config.json"))
VAL_PATH  = find("val.parquet")
TEST_PATH = find("test.csv")
INFO      = json.load(open(find("train_info.json")))

# Hidden test rerun-la Kaggle intha env var set pannum -> benchmarks skip
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
N_GPU = torch.cuda.device_count()

print("model:", MODEL_DIR)
print("train info:", INFO)
print("RERUN:", RERUN, "| GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
print("torch", torch.__version__)

In [ ]:
N_BENCH = 1000                              # benchmark-ku evvalavu val samples
MAX_P, MAX_R = INFO["max_p"], INFO["max_r"]  # training-la use panna same truncation
USE_TTA = False                             # benchmark paathu apram decide pannalam

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
PAD_ID = tok.pad_token_id

def parse(x):
    try:
        v = json.loads(x)
        if isinstance(v, list):
            return "\n".join("" if s is None else str(s) for s in v)
        return str(v)
    except Exception:
        return str(x)

H_P = tok("Prompt: ", add_special_tokens=False)["input_ids"]
H_A = tok("\n\nResponse A: ", add_special_tokens=False)["input_ids"]
H_B = tok("\n\nResponse B: ", add_special_tokens=False)["input_ids"]
H_Q = tok("\n\nWhich response is better?", add_special_tokens=False)["input_ids"]

def build(d, swap=False):
    a_col, b_col = ("response_b", "response_a") if swap else ("response_a", "response_b")
    P = tok(d["prompt"].tolist(), add_special_tokens=False, truncation=True, max_length=MAX_P)["input_ids"]
    A = tok(d[a_col].tolist(),    add_special_tokens=False, truncation=True, max_length=MAX_R)["input_ids"]
    B = tok(d[b_col].tolist(),    add_special_tokens=False, truncation=True, max_length=MAX_R)["input_ids"]
    return [H_P + p + H_A + a + H_B + b + H_Q for p, a, b in zip(P, A, B)]

val_df  = pd.read_parquet(VAL_PATH).iloc[:N_BENCH]   # training notebook-la already parse aagiduchu
vids    = build(val_df)
vids_sw = build(val_df, swap=True)
vy      = val_df["label"].values
print("val samples:", len(vids), "| mean len:", int(np.mean([len(x) for x in vids])))

In [ ]:
from tqdm.auto import tqdm

# Dynamic padding: ovvoru batch-um athoda longest sequence varaikum mattum pad
def collate(seqs, device):
    L = max(len(s) for s in seqs)
    ids  = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)]  = torch.tensor(s)
        mask[i, :len(s)] = 1
    return {"input_ids": ids.to(device), "attention_mask": mask.to(device)}

@torch.inference_mode()
def predict(model, ids, bs=16, sort=True, device="cuda:0", progress=True):
    # sort=True: similar length samples onna batch -> padding waste kammi
    order = np.argsort([-len(x) for x in ids], kind="stable") if sort else np.arange(len(ids))
    out = np.zeros((len(ids), 3), dtype=np.float32)
    starts = range(0, len(ids), bs)
    if progress:
        starts = tqdm(starts, leave=False)          # progress bar: odutha illaya nu theriyum
    with torch.cuda.device(device):
        for i in starts:
            idx = order[i:i + bs]
            batch = collate([ids[j] for j in idx], device)
            logits = model(**batch, use_cache=False).logits.float()
            out[idx] = torch.softmax(logits, -1).cpu().numpy()
    return out

def predict_2gpu(models, ids, bs=16):
    # Data parallel: test set-a rendu half aakki, ovvoru GPU-layum oru model copy
    half = len(ids) // 2
    parts, res = [ids[:half], ids[half:]], [None, None]
    def run(k):
        res[k] = predict(models[k], parts[k], bs=bs, device=f"cuda:{k}", progress=False)
    threads = [threading.Thread(target=run, args=(k,)) for k in range(2)]
    for t in threads: t.start()
    for t in threads: t.join()
    return np.concatenate(res)

def load(dtype=torch.float16, device="cuda:0", **kw):
    # device_map: model-a nera antha ore GPU-la load (Transformers 5 split pannaama irukka)
    kw.setdefault("device_map", {"": device})
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, dtype=dtype, **kw)
    print("loaded on:", getattr(m, "hf_device_map", None))
    return m.eval()

def model_gb(m):
    return round(sum(t.numel() * t.element_size() for t in list(m.parameters()) + list(m.buffers())) / 1e9, 2)

results = []
def bench(name, fn, model=None):
    for d in range(N_GPU):
        torch.cuda.synchronize(d); torch.cuda.reset_peak_memory_stats(d)
    t = time.time()
    p = fn()
    for d in range(N_GPU):
        torch.cuda.synchronize(d)
    dt = time.time() - t
    ll = log_loss(vy, p, labels=[0, 1, 2]) if np.isfinite(p).all() else float("nan")
    r = dict(variant=name, seconds=round(dt, 1), samples_per_sec=round(len(vy) / dt, 1),
             model_gb=model_gb(model) if model is not None else None,
             peak_vram_gb=round(max(torch.cuda.max_memory_allocated(d) for d in range(N_GPU)) / 1e9, 2),
             val_log_loss=round(ll, 4))
    results.append(r)
    print(r)
    return p

## Benchmark 1: fp32 naive baseline + bitsandbytes INT8 / INT4
GPU-la vera model illaama thani-a run pannrom, appo thaan VRAM numbers correct-a varum.

In [ ]:
if not RERUN:
    m32 = load(torch.float32)
    bench("fp32 bs1 (naive)", lambda: predict(m32, vids, bs=1, sort=False), m32)
    del m32; torch.cuda.empty_cache()

    try:
        try:
            import bitsandbytes
        except ImportError:
            whl = glob.glob("/kaggle/input/**/wheels/bitsandbytes*.whl", recursive=True)
            subprocess.run(["pip", "install", "-q", "--no-index", "--no-deps", whl[0]], check=True)
            import bitsandbytes
        print("bitsandbytes", bitsandbytes.__version__)
        from transformers import BitsAndBytesConfig
        qcfgs = {
            "int8 bnb bs16 sorted": BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=["score"]),
            "int4 nf4 bnb bs16 sorted": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16,
                                                           llm_int8_skip_modules=["score"]),
        }
        for name, q in qcfgs.items():
            try:
                mq = load(quantization_config=q, device_map={"": 0})
                bench(name, lambda: predict(mq, vids, bs=16), mq)
                del mq; torch.cuda.empty_cache()
            except Exception as e:
                print("SKIP", name, "->", repr(e)[:300])
    except Exception as e:
        print("bitsandbytes section skip ->", repr(e)[:300])

## Final model load (fp16, ovvoru GPU-kkum oru copy)
Intha cell eppavume odum (submission rerun-laiyum).

In [ ]:
m16 = load(torch.float16, device="cuda:0")
m16_b = load(torch.float16, device="cuda:1") if N_GPU > 1 else None
print("fp16 model size:", model_gb(m16), "GB per GPU")

## Benchmark 2: fp16, batching, length sorting, swap TTA, 2 GPU

In [ ]:
if not RERUN:
    bench("fp16 bs1", lambda: predict(m16, vids, bs=1, sort=False), m16)
    bench("fp16 bs16", lambda: predict(m16, vids, bs=16, sort=False), m16)
    bench("fp16 bs16 sorted", lambda: predict(m16, vids, bs=16, sort=True), m16)
    bench("fp16 bs32 sorted", lambda: predict(m16, vids, bs=32, sort=True), m16)

    def tta():
        p1 = predict(m16, vids, bs=16)
        p2 = predict(m16, vids_sw, bs=16)[:, [1, 0, 2]]   # swap panna A<->B probabilities thirumba maathanum
        return (p1 + p2) / 2
    bench("fp16 bs16 sorted + swap-TTA", tta, m16)

    if m16_b is not None:
        bench("fp16 bs16 sorted 2xT4", lambda: predict_2gpu([m16, m16_b], vids), m16)

## Results table + speed-vs-accuracy chart

In [ ]:
if not RERUN:
    import matplotlib.pyplot as plt
    res = pd.DataFrame(results)
    res["speedup_vs_naive"] = (res["seconds"].iloc[0] / res["seconds"]).round(1)
    res.to_csv("benchmark_results.csv", index=False)
    display(res)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(res["seconds"], res["val_log_loss"])
    for _, r in res.iterrows():
        ax.annotate(r["variant"], (r["seconds"], r["val_log_loss"]), fontsize=8,
                    xytext=(4, 4), textcoords="offset points")
    ax.set_xscale("log")
    ax.set_xlabel(f"seconds for {len(vy)} val samples (log scale, kammi = fast)")
    ax.set_ylabel("val log loss (kammi = better)")
    ax.set_title(f"Speed vs accuracy: Qwen2.5-1.5B classifier on Kaggle {torch.cuda.get_device_name(0)}")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("benchmark.png", dpi=150)
    plt.show()

## Test prediction + submission.csv
`USE_TTA = True` aakkanum-na benchmark-la swap-TTA log loss clear-a better-a irukanum (runtime 2x aagum).

In [ ]:
test = pd.read_csv(TEST_PATH)
for c in ["prompt", "response_a", "response_b"]:
    test[c] = test[c].apply(parse)

def run_all(ids):
    return predict_2gpu([m16, m16_b], ids) if m16_b is not None else predict(m16, ids)

t = time.time()
probs = run_all(build(test))
if USE_TTA:
    probs = (probs + run_all(build(test, swap=True))[:, [1, 0, 2]]) / 2

sub = pd.DataFrame({"id": test["id"], "winner_model_a": probs[:, 0],
                    "winner_model_b": probs[:, 1], "winner_tie": probs[:, 2]})
sub.to_csv("submission.csv", index=False)
print(sub.shape, f"| test inference {time.time() - t:.1f}s | TTA={USE_TTA}")
sub.head()